<a href="https://colab.research.google.com/github/tonytang1020-source/daily_stock_analysis/blob/main/6_0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import time
import requests
import datetime
import numpy as np
import os
import sys
from functools import wraps

# ==========================================
# 战术参数配置区
# ==========================================
TARGET_STOCK = "sz002602"
DYNAMIC_LOOKBACK_PERIOD = 30      # 原参数保留，实际会被动态窗口替代
VOLUME_CONFIRM_MULTIPLIER = 3.5
KDJ_UPPER_THRESHOLD = 82
KDJ_LOWER_THRESHOLD = 25

# [新增] 动态窗口相关参数
ATR_PERIOD = 14                    # ATR计算周期
BASE_WINDOW = 20                    # 基础窗口
MIN_WINDOW = 10                      # 最小窗口
MAX_WINDOW = 60                      # 最大窗口

# [新增] 数据获取容错
MAX_RETRIES = 3
RETRY_DELAY = 2

# 日志文件
LOG_FILE = f"Log_{TARGET_STOCK}_{datetime.datetime.now().strftime('%Y%m%d')}.txt"

# ==========================================
# 工具函数：重试装饰器
# ==========================================
def retry(max_retries=MAX_RETRIES, delay=RETRY_DELAY):
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            for i in range(max_retries):
                try:
                    return func(*args, **kwargs)
                except Exception as e:
                    if i == max_retries - 1:
                        raise
                    time.sleep(delay)
            return None
        return wrapper
    return decorator

def write_blackbox(msg):
    """战地黑匣子：将关键数据写入本地TXT"""
    with open(LOG_FILE, "a", encoding="utf-8") as f:
        f.write(f"[{datetime.datetime.now().strftime('%H:%M:%S')}] {msg}\n")

# ==========================================
# 数据获取函数（带重试）
# ==========================================
@retry(max_retries=3, delay=2)
def get_historical_klines(stock_code, datalen=60):
    """获取历史K线，返回已完成K线数据（排除当前未闭合的最后一根）"""
    url = f"http://ifzq.gtimg.cn/appstock/app/kline/mkline?param={stock_code},m1,,{datalen}"
    response = requests.get(url, timeout=5)
    data = response.json()
    k_list = data['data'][stock_code]['m1']

    # 腾讯接口最后一条可能是当前未闭合K线，我们只取前datalen-1条作为历史
    # 但为了安全，检查时间：如果当前分钟数大于最后一条的分钟，说明最后一条已闭合，否则剔除
    # 简化处理：总是剔除最后一条，因为获取时机保证在每分钟初
    hist_list = k_list[:-1]  # 去掉最后一条未闭合的

    closes = np.array([float(k[2]) for k in hist_list])
    highs = np.array([float(k[3]) for k in hist_list])
    lows = np.array([float(k[4]) for k in hist_list])
    volumes = np.array([float(k[5]) * 100 for k in hist_list])
    return closes, highs, lows, volumes

@retry(max_retries=3, delay=2)
def get_sina_real_time_data(stock_code):
    url = f"http://hq.sinajs.cn/list={stock_code}"
    headers = {'Referer': 'https://finance.sina.com.cn'}
    response = requests.get(url, headers=headers, timeout=3)
    data_list = response.text.split('"')[1].split(',')
    curr_p = float(data_list[3])
    yest_c = float(data_list[2])
    total_v = float(data_list[8])
    vwap = (float(data_list[9])/total_v) if total_v > 0 else curr_p
    return curr_p, total_v, vwap, yest_c

# ==========================================
# 技术指标计算（基于已完成K线）
# ==========================================
def EMA(data, n):
    ema = np.zeros_like(data)
    ema[0] = data[0]
    alpha = 2/(n+1)
    for i in range(1, len(data)):
        ema[i] = alpha * data[i] + (1-alpha) * ema[i-1]
    return ema

def calculate_all_indicators(closes, highs, lows):
    """
    输入：已完成K线的价格序列（至少30根）
    返回：macd, diff, k, j, rsi （均基于最后已完成的一根K线）
    """
    if len(closes) < 30:
        return 0, 0, 50, 50, 50

    # MACD
    ema12 = EMA(closes, 12)
    ema26 = EMA(closes, 26)
    diff = ema12 - ema26
    dea = EMA(diff, 9)
    macd = (diff - dea) * 2

    # KDJ (基于最后9根已完成K线)
    rsv = np.zeros_like(closes)
    k = np.zeros_like(closes)
    d = np.zeros_like(closes)
    k[0], d[0] = 50, 50
    for i in range(8, len(closes)):
        h9 = np.max(highs[i-8:i+1])
        l9 = np.min(lows[i-8:i+1])
        if h9 != l9:
            rsv[i] = (closes[i] - l9) / (h9 - l9) * 100
        else:
            rsv[i] = 0
        if i == 8:
            k[i] = rsv[i]
            d[i] = rsv[i]
        else:
            k[i] = (2/3)*k[i-1] + (1/3)*rsv[i]
            d[i] = (2/3)*d[i-1] + (1/3)*k[i]
    j = 3*k - 2*d

    # RSI (6日)
    diff_p = np.diff(closes)
    up = np.sum(diff_p[-6:][diff_p[-6:] > 0])
    down = abs(np.sum(diff_p[-6:][diff_p[-6:] < 0]))
    rsi = (up/(up+down)*100) if (up+down) > 0 else 50

    return macd[-1], diff[-1], k[-1], j[-1], rsi

def calculate_atr(highs, lows, closes, period=14):
    """计算ATR序列，返回最新ATR值及过去N根平均ATR"""
    if len(highs) < period + 1:
        return 0, 0
    tr = np.zeros(len(highs))
    for i in range(1, len(highs)):
        hl = highs[i] - lows[i]
        hc = abs(highs[i] - closes[i-1])
        lc = abs(lows[i] - closes[i-1])
        tr[i] = max(hl, hc, lc)
    atr = np.zeros(len(highs))
    atr[period-1] = np.mean(tr[1:period])  # 第一个ATR用简单平均
    for i in range(period, len(highs)):
        atr[i] = (atr[i-1] * (period-1) + tr[i]) / period
    return atr[-1], np.mean(atr[max(0, len(atr)-60):])

def get_dynamic_support_resistance(highs, lows, closes, base_window=BASE_WINDOW, atr_period=ATR_PERIOD):
    """
    根据ATR动态调整支撑/阻力计算窗口
    返回：(支撑, 阻力)
    """
    if len(highs) < max(base_window, atr_period+1):
        return np.min(lows), np.max(highs)

    current_atr, avg_atr = calculate_atr(highs, lows, closes, atr_period)
    if avg_atr == 0 or current_atr == 0:
        window = base_window
    else:
        # 窗口长度与ATR成反比：波动大时窗口缩短，波动小时窗口拉长
        ratio = avg_atr / current_atr
        window = int(base_window * ratio)
        window = max(MIN_WINDOW, min(MAX_WINDOW, window))

    # 取最近window根已完成K线的最高最低
    support = np.min(lows[-window:])
    resistance = np.max(highs[-window:])
    return support, resistance, window

# ==========================================
# 信号触发
# ==========================================
def trigger_alert(stock_code, action, price, info):
    msg = f"🚨 {action} | 现价:{price:.2f} | {info}"
    print(f"\n[{datetime.datetime.now().strftime('%H:%M:%S')}] {msg}\n" + "="*70)
    write_blackbox(f"【交易指令】{msg}")

# ==========================================
# 实盘主函数
# ==========================================
def intraday_sniper_bot():
    print(f"启动量化狙击引擎 V6.0 (自适应窗口+防御性升级) | 标的: {TARGET_STOCK}")
    print(f"日志路径: {os.path.abspath(LOG_FILE)}")

    # 初始化历史数据
    c, h, l, v = None, None, None, None
    while c is None:
        try:
            c, h, l, v = get_historical_klines(TARGET_STOCK, datalen=60)
        except:
            time.sleep(1)
    print("✅ 引擎并网！监控数据将同步存盘。\n" + "-"*80)

    last_kline_update = 0  # 用于控制K线获取时机

    while True:
        try:
            # 获取实时数据
            curr_p, total_v, vwap, yest_c = get_sina_real_time_data(TARGET_STOCK)
            if curr_p is None:
                time.sleep(3)
                continue

            # 每分钟的第5秒更新一次K线（确保上分钟K线已闭合）
            now = datetime.datetime.now()
            if now.second < 10 and time.time() - last_kline_update > 50:  # 避免重复更新
                try:
                    res = get_historical_klines(TARGET_STOCK, datalen=60)
                    if res[0] is not None:
                        c, h, l, v = res
                        last_kline_update = time.time()
                except Exception as e:
                    write_blackbox(f"K线更新失败: {e}")

            # 检查历史数据是否足够
            if c is None or len(c) < 30:
                time.sleep(3)
                continue

            # 计算指标（基于已完成K线）
            macd, diff, k, j, rsi = calculate_all_indicators(c, h, l)

            # 动态支撑/阻力
            d_sup, d_res, used_window = get_dynamic_support_resistance(h, l, c)

            # 量比（使用最近一根已完成K线的成交量与过去30根平均对比）
            avg_v = np.mean(v[-30:]) if len(v) >= 30 else np.mean(v)
            vol_r = v[-1] / avg_v if avg_v > 0 else 1

            # 实时涨跌幅、乖离
            change_pct = (curr_p - yest_c) / yest_c * 100
            bias = (curr_p - vwap) / vwap * 100

            # 信号判断
            status = "🛡️ 稳健震荡"
            signal_triggered = False

            # 低吸条件
            if curr_p <= d_sup and k < KDJ_LOWER_THRESHOLD and rsi < 30:
                if macd > 0 and vol_r > VOLUME_CONFIRM_MULTIPLIER:
                    trigger_alert(TARGET_STOCK, "🎯 极限低吸", curr_p,
                                 f"KDJ:{k:.1f} RSI:{rsi:.1f} 量比:{vol_r:.1f}x 窗口:{used_window}")
                    signal_triggered = True
                else:
                    status = "👀 探底中(等金叉/放量)"

            # 高抛条件
            elif curr_p >= d_res and k > KDJ_UPPER_THRESHOLD and rsi > 70:
                if macd < 0 and vol_r > VOLUME_CONFIRM_MULTIPLIER:
                    trigger_alert(TARGET_STOCK, "🏹 极速高抛", curr_p,
                                 f"KDJ:{k:.1f} RSI:{rsi:.1f} 量比:{vol_r:.1f}x 窗口:{used_window}")
                    signal_triggered = True
                else:
                    status = "👀 冲高中(等死叉/放量)"

            # 如果触发了信号，休眠60秒避免连续触发
            if signal_triggered:
                time.sleep(60)
                continue

            # 控制台输出
            t_now = now.strftime('%H:%M:%S')
            line = (f"[{t_now}] {curr_p:.2f} (涨跌:{change_pct:+.2f}% | 乖离:{bias:+.2f}%) "
                   f"| 支:{d_sup:.2f} 阻:{d_res:.2f}(窗:{used_window}) | 量:{vol_r:.1f}x "
                   f"| KDJ:{k:.0f} RSI:{rsi:.0f} | {status}")
            print(line.ljust(120), end="\r")

            # 每5分钟记录一次心跳
            if int(now.timestamp()) % 300 < 3:  # 粗略每5分钟
                write_blackbox(line.strip())

            time.sleep(3)

        except KeyboardInterrupt:
            print("\n🛑 狙击引擎终止")
            write_blackbox("引擎手动终止")
            break
        except Exception as e:
            write_blackbox(f"主循环异常: {e}")
            time.sleep(5)

# ==========================================
# 简易回测功能
# ==========================================
def run_backtest(data_file):
    """
    回测模式：需要准备历史1分钟K线CSV文件，包含列：
    datetime, open, high, low, close, volume
    """
    import pandas as pd
    print("启动回测模式...")
    df = pd.read_csv(data_file, parse_dates=['datetime'])
    df = df.sort_values('datetime')

    # 模拟逐根K线
    positions = []  # 记录交易：时间，方向，价格，原因
    in_position = False
    buy_price = 0

    for i in range(60, len(df)):  # 从第60根开始，确保有足够历史
        # 取到当前K线为止的历史数据（不包括未来）
        hist = df.iloc[:i+1]  # 包含当前K线，但指标计算要基于前一根已闭合
        current = df.iloc[i]

        # 用前一根及之前的数据计算指标（避免未来函数）
        closes = hist['close'].values[:-1]
        highs = hist['high'].values[:-1]
        lows = hist['low'].values[:-1]
        volumes = hist['volume'].values[:-1]

        if len(closes) < 30:
            continue

        # 计算指标
        macd, diff, k, j, rsi = calculate_all_indicators(closes, highs, lows)

        # 动态支撑/阻力
        d_sup, d_res, used_window = get_dynamic_support_resistance(highs, lows, closes)

        # 量比
        avg_v = np.mean(volumes[-30:])
        vol_r = volumes[-1] / avg_v if avg_v > 0 else 1

        # 当前K线的收盘价（模拟实时价格）
        curr_p = current['close']
        yest_c = df.iloc[i-1]['close']  # 前一日收盘，简化处理，实际应为前日收盘
        change_pct = (curr_p - yest_c) / yest_c * 100

        # 信号判断
        signal = None
        if curr_p <= d_sup and k < KDJ_LOWER_THRESHOLD and rsi < 30:
            if macd > 0 and vol_r > VOLUME_CONFIRM_MULTIPLIER:
                signal = 'BUY'
        elif curr_p >= d_res and k > KDJ_UPPER_THRESHOLD and rsi > 70:
            if macd < 0 and vol_r > VOLUME_CONFIRM_MULTIPLIER:
                signal = 'SELL'

        # 模拟交易（简单开平仓，不考虑持仓数量）
        if signal == 'BUY' and not in_position:
            positions.append({
                'datetime': current['datetime'],
                'action': 'BUY',
                'price': curr_p,
                'reason': f'KDJ:{k:.1f} RSI:{rsi:.1f} 量比:{vol_r:.1f}'
            })
            buy_price = curr_p
            in_position = True
        elif signal == 'SELL' and in_position:
            positions.append({
                'datetime': current['datetime'],
                'action': 'SELL',
                'price': curr_p,
                'profit': (curr_p - buy_price)/buy_price*100,
                'reason': f'KDJ:{k:.1f} RSI:{rsi:.1f} 量比:{vol_r:.1f}'
            })
            in_position = False

    # 输出回测结果
    if positions:
        df_pos = pd.DataFrame(positions)
        print("\n=== 回测交易记录 ===")
        print(df_pos)
        if 'profit' in df_pos.columns:
            wins = df_pos[df_pos['profit'] > 0]
            print(f"\n胜率: {len(wins)}/{len(df_pos)} = {len(wins)/len(df_pos)*100:.1f}%")
            print(f"平均盈利: {df_pos['profit'].mean():.2f}%")
    else:
        print("无交易信号")

# ==========================================
# 入口：根据参数选择模式
# ==========================================
if __name__ == "__main__":
    if len(sys.argv) > 1 and sys.argv[1] == '--backtest':
        if len(sys.argv) > 2:
            run_backtest(sys.argv[2])
        else:
            print("请指定回测数据文件路径，例如: python script.py --backtest history.csv")
    else:
        intraday_sniper_bot()